# Phase 4: VIEW + 데이터마트
**DB:** `data/franchise.db` (맛나국밥 가상 프랜차이즈)

| 단계 | 내용 |
|------|------|
| 1 | VIEW 생성 — 자주 쓰는 쿼리를 테이블처럼 저장 |
| 2 | VIEW 활용 — 단순 SELECT로 복잡한 집계 재사용 |

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/franchise.db')

def query(sql):
    return pd.read_sql_query(sql, conn)

def execute(sql):
    conn.executescript(sql)
    print('완료')

---
## 1. VIEW 생성
### 과제 4-1: v_store_sales
매장별 총매출 + 순위를 담은 VIEW  
> `CREATE VIEW 뷰이름 AS SELECT ...`  
> 이후엔 `SELECT * FROM v_store_sales` 로 재사용 가능

In [2]:
execute("""
DROP VIEW IF EXISTS v_store_sales;
CREATE VIEW v_store_sales AS
SELECT
    region,
    store_type,
    store_name,
    SUM(total_amount) AS 총매출,
    RANK() OVER (ORDER BY SUM(total_amount) DESC) AS 순위
FROM daily_sales
JOIN stores ON daily_sales.store_id = stores.store_id
GROUP BY store_name, region, store_type
ORDER BY 총매출 DESC;
""")

완료


In [3]:
query("SELECT * FROM v_store_sales")

,region,store_type,store_name,총매출,순위
0,서울,직영,맛나국밥 영등포구점,654547000,1
1,경기,가맹,맛나국밥 용인시점,629635000,2
2,경기,가맹,맛나국밥 고양시점,620811000,3
3,서울,가맹,맛나국밥 강남구점,604537000,4
4,대구,직영,맛나국밥 달서구점,601874000,5
5,서울,직영,맛나국밥 강서구점,589305000,6
6,서울,가맹,맛나국밥 노원구점,580658000,7
7,서울,가맹,맛나국밥 종로구점,557547000,8
8,부산,가맹,맛나국밥 해운대구점,535210000,9
9,서울,가맹,맛나국밥 서초구점,521674000,10


### 과제 4-3: v_monthly_category_sales
월별 카테고리별 매출/수량을 담은 VIEW  
> 데이터마트 — 분석 목적에 맞게 미리 집계해둔 테이블

In [4]:
execute("""
DROP VIEW IF EXISTS v_monthly_category_sales;
CREATE VIEW v_monthly_category_sales AS
SELECT
    category,
    SUBSTR(sale_date, 1, 7) AS 월,
    SUM(total_amount) AS 총매출,
    SUM(quantity) AS 총수량
FROM daily_sales
JOIN menu_items ON daily_sales.menu_id = menu_items.menu_id
GROUP BY 월, category
ORDER BY 월, category;
""")

완료


In [5]:
query("SELECT * FROM v_monthly_category_sales")

,category,월,총매출,총수량
0,사이드,2025-01,101769000,25407
1,안주,2025-01,119532000,4269
2,주류,2025-01,78340000,14232
3,탕,2025-01,710037000,53266
4,사이드,2025-02,93172000,23380
5,안주,2025-02,111384000,3978
6,주류,2025-02,71427000,12969
7,탕,2025-02,646894000,48484
8,사이드,2025-03,103398000,25932
9,안주,2025-03,121800000,4350


---
## 2. VIEW 활용
### 과제 4-2: 직영점만 조회
VIEW를 테이블처럼 WHERE로 필터링  
> JOIN/GROUP BY 없이 단순 SELECT만으로 복잡한 집계 결과 재사용

In [6]:
query("""
SELECT store_name, region, 총매출, 순위
FROM v_store_sales
WHERE store_type = '직영'
ORDER BY 순위
""")

,store_name,region,총매출,순위
0,맛나국밥 영등포구점,서울,654547000,1
1,맛나국밥 달서구점,대구,601874000,5
2,맛나국밥 강서구점,서울,589305000,6
3,맛나국밥 성남시점,경기,414822000,20


### 월별 탕 카테고리 매출 추이

In [7]:
query("""
SELECT 월, 총매출, 총수량
FROM v_monthly_category_sales
WHERE category = '탕'
ORDER BY 월
""")

,월,총매출,총수량
0,2025-01,710037000,53266
1,2025-02,646894000,48484
2,2025-03,579300000,43430
3,2025-04,556082000,41709
4,2025-05,574511000,43071
5,2025-06,561503000,42089
6,2025-07,482355000,36186
7,2025-08,492203000,36900
8,2025-09,551159000,41334
9,2025-10,573376000,42983


In [8]:
conn.close()